In [185]:
from sklearn import datasets
import numpy as np

In [187]:
iris_d = datasets.load_iris()
list(iris_d.keys())

['data',
 'target',
 'frame',
 'target_names',
 'DESCR',
 'feature_names',
 'filename',
 'data_module']

In [189]:
X = iris_d["data"][:,(2,3)] #petal l, petal w

In [191]:
y = iris_d["target"]

In [193]:
# Добавляем столбец смещения
X_with_bias = np.c_[np.ones([len(X), 1]), X]

In [195]:
np.random.seed(2042)

In [197]:
# Разделение данных на тренировочный, тестовый и проверочный наборы
test_r = 0.1
val_r = 0.1
total_size = len(X_with_bias)

test_size = int(test_r * total_size)
val_size = int(val_r * total_size)
train_size = total_size - test_size - val_size

random_indices = np.random.permutation(total_size)

X_train = X_with_bias[random_indices[:train_size]]
y_train = y[random_indices[:train_size]]

X_valid = X_with_bias[random_indices[train_size:-test_size]]
y_valid = y[random_indices[train_size:-test_size]]

X_test =  X_with_bias[random_indices[-test_size:]]
y_test = y[random_indices[-test_size:]]

In [199]:
# функция для преобразования вектора индексов классов в матрицу с вектором one-hot для каждого экземпляра
def to_one_hot(y):
    n_unique_classes = y.max() + 1
    n = len(y)
    One_hot_matrix = np.zeros((n, n_unique_classes))
    One_hot_matrix[np.arange(n), y] = 1
    return One_hot_matrix

In [201]:
y_train_one_hot = to_one_hot(y_train)
y_valid_one_hot = to_one_hot(y_valid)
y_test_one_hot = to_one_hot(y_test)

In [203]:
# реализация собственной функции softmax
def softmax(logits):
    exps_all_logits = np.exp(logits)
    exps_sum = np.sum(exps_all_logits, axis=1, keepdims=True)
    return exps_all_logits/exps_sum

In [205]:
n_inputs = X_train.shape[1]

In [207]:
n_outputs = y.max() + 1

In [209]:
eta = 0.01
n_iter = 5001
n = len(X_train)
epsilon = 1e-7

Theta = np.random.randn(n_inputs, n_outputs)

for i in range(n_iter):
    logits = X_train.dot(Theta)
    y_proba = softmax(logits)
    if i % 500 == 0:
        loss = -np.mean(np.sum(y_train_one_hot * np.log(y_proba + epsilon), axis = 1))
        print(i, loss)
    error = y_proba - y_train_one_hot
    gradients = 1 / n * X_train.T.dot(error)
    Theta = Theta - eta * gradients

0 5.452321039208442
500 0.8410408731887224
1000 0.6946593279048037
1500 0.6070679262204142
2000 0.5490074133467334
2500 0.5072047195660898
3000 0.47522821181042835
3500 0.44966847695113565
4000 0.42856239935240764
4500 0.41069827109493157
5000 0.39528435883051866


In [211]:
Theta

array([[ 3.28234296, -0.62552812, -2.98577763],
       [-1.18327527,  0.15727818,  0.07628774],
       [-0.72176685, -0.190768  ,  1.59365252]])

In [213]:
# Проверка на проверочном наборе
logits = X_valid.dot(Theta)
y_proba = softmax(logits)
y_predict = np.argmax(y_proba, axis=1)

accuracy_score = np.mean(y_predict == y_valid)
accuracy_score

0.9333333333333333

In [215]:
# добавляем штрафы и изменяем скорость обучения
eta = 0.1
n_iter = 5001
n = len(X_train)
epsilon = 1e-7
alpha = 0.1

Theta = np.random.randn(n_inputs, n_outputs)

for i in range(n_iter):
    logits = X_train.dot(Theta)
    y_proba = softmax(logits)
    if i % 500 == 0:
        xentropy_loss = -np.mean(np.sum(y_train_one_hot * np.log(y_proba + epsilon), axis = 1))
        l2_loss = 1/2 * np.sum(np.square(Theta[1:]))
        loss = xentropy_loss + alpha * l2_loss
        print(i, loss)
    error = y_proba - y_train_one_hot
    gradients = 1 / n * X_train.T.dot(error) + np.r_[np.zeros([1, n_outputs]), alpha * Theta[1:]]
    Theta = Theta - eta * gradients

0 6.639346607419499
500 0.5428224426347683
1000 0.5130493773818743
1500 0.5045159164371424
2000 0.5013663136676507
2500 0.5000991026026991
3000 0.4995668173342027
3500 0.49933766758441805
4000 0.49923753199615256
4500 0.49919335974230783
5000 0.4991737556995439


In [217]:
# Проверка на проверочном наборе
logits = X_valid.dot(Theta)
y_proba = softmax(logits)
y_predict = np.argmax(y_proba, axis=1)

accuracy_score = np.mean(y_predict == y_valid)
accuracy_score

0.9333333333333333

In [219]:
# добавление ранней остановки
eta = 0.1 
n_iterations = 5001
m = len(X_train)
epsilon = 1e-7
alpha = 0.1 
best_loss = np.infty

Theta = np.random.randn(n_inputs, n_outputs)

for i in range(n_iterations):
    logits = X_train.dot(Theta)
    y_proba = softmax(logits)
    error = y_proba - y_train_one_hot
    gradients = 1/m * X_train.T.dot(error) + np.r_[np.zeros([1, n_outputs]), alpha * Theta[1:]]
    Theta = Theta - eta * gradients

    logits = X_valid.dot(Theta)
    y_proba = softmax(logits)
    xentropy_loss = -np.mean(np.sum(y_valid_one_hot * np.log(y_proba + epsilon), axis=1))
    l2_loss = 1/2 * np.sum(np.square(Theta[1:]))
    loss = xentropy_loss + alpha * l2_loss
    if i % 500 == 0:
        print(i, loss)
    if loss < best_loss:
        best_loss = loss
    else:
        print(i - 1, best_loss)
        print(i, loss, "early stopping!")
        break

0 5.271588529057238
500 0.5732119193897453
1000 0.526118409097518
1500 0.5099681171563867
2000 0.502762643707622
2500 0.49911837147257904
3000 0.497119838338226
3500 0.49595878620419975
4000 0.4952553066732563
4500 0.4948158773904282
5000 0.4945353464901719


In [221]:
# Проверка на проверочном наборе
logits = X_valid.dot(Theta)
y_proba = softmax(logits)
y_predict = np.argmax(y_proba, axis=1)

accuracy_score = np.mean(y_predict == y_valid)
accuracy_score

0.9333333333333333

In [223]:
# Проверка на тестовом наборе
logits = X_test.dot(Theta)
y_proba = softmax(logits)
y_predict = np.argmax(y_proba, axis=1)

accuracy_score = np.mean(y_predict == y_test)
accuracy_score

0.9333333333333333